In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import ast
import re

# Load Databooks

In [0]:
# Define the base directory generically
DATA_DIR = Path('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-analysis-datasets')

In [0]:
# Model Labels
df = pd.read_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-classification/exercises_dataset_CLEAN.csv')

In [0]:
# Detailed Exercise Program Dataset
prog = pd.read_csv('/Volumes/projects/default/exercise-analysis/programs_detailed_boostcamp_kaggle.csv')


# Add on the labels to the Detailed Exercise Program Dataset

In [0]:
def norm(s):
    s = str(s).lower()
    s = re.sub(r'\(.*?\)', '', s)          # strip equipment notes in parens
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

## Load the established labels into the exercise list

In [0]:
# Labeled - Normalised Exercise Label Dataset
labeled = df.copy(deep = True)
labeled['norm'] = labeled['exercise'].apply(norm)

# Unique Names - Normalised Exercise Program Dataset (TO LABEL)
unique_names = pd.DataFrame({'exercise_name': prog['exercise_name'].unique()})
unique_names['norm'] = unique_names['exercise_name'].apply(norm)

In [0]:
lookup = labeled.drop_duplicates('norm').set_index('norm')['label']

labeled['label'] = labeled['norm'].map(lookup)
unique_names['label'] = unique_names['norm'].map(lookup) # Apply the original labelled dataset, labeled, to unique_names

## Load in the labels from the text_classification model

In [0]:
# The dataset containing missing data labels
to_predict = pd.read_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-classification/exercises_to_predict_MANUAL_LABELED.csv')

In [0]:
# Create a dictionary to match the exercise program labels with the exercise labels
to_predict['final_label'] = to_predict['manual_label']
to_predict['final_label'] = to_predict['final_label'].fillna(to_predict['my_label'])

label_dict = dict(zip(to_predict['exercise_name'], to_predict['final_label']))

In [0]:
unique_names['model_labels'] = unique_names['exercise_name'].map(label_dict) #Apply the predicted names from to_predict
unique_names['label'] = unique_names['label'].fillna(unique_names['model_labels'])

program_label = dict(zip(unique_names['exercise_name'], unique_names['label']))
prog['exercise_label'] = prog['exercise_name'].map(program_label)

## Remove Extra Programs (with missing info)

In [0]:
prog[prog.equipment.isna()].title.unique() #Identify programs with no equipment label

In [0]:
prog.drop(prog[prog.title == 'Lera'].index, inplace=True) #Remove these programs

# Separate programs with duplicate titles (via description)

In [0]:
dup_titles = prog.groupby('title')['description'].nunique().sort_values(ascending=False)
dup_titles[dup_titles > 1] # Identify programs with duplicate program titles

In [0]:
duplicate_titles = list(dup_titles[dup_titles > 1].index)

In [0]:
# Create a new DataFrame with the duplicate titles
prog2 = pd.DataFrame()
for title in duplicate_titles:
    desc_list = prog[(prog.title == title)].description.unique()
    tmp_title = 1
    for desc in desc_list:
        tmp = prog[(prog.title == title) & (prog.description == desc)]
        tmp['title'] = title + f' ({tmp_title})'
        prog2 = pd.concat([prog2, tmp])
        tmp_title += 1


In [0]:
# Drop the original duplicate title rows 
prog.drop(prog[prog.title.isin(duplicate_titles)].index, inplace=True)

In [0]:
# Replace with the same rows (with new titles)
prog = pd.concat([prog, prog2])
prog.reset_index(inplace=True)

prog.drop(columns=['index'], inplace=True)

# Clean databook, adjust the level and goal columns

In [0]:
prog.level = prog.level.apply(ast.literal_eval)
prog.goal = prog.goal.apply(ast.literal_eval)

In [0]:
prog.drop(prog[prog.title == 'Mania (Upper/Lower)'].index, inplace = True)
prog.drop(prog[prog.title == 'Viking strong'].index, inplace = True)

In [0]:
level_map = {'Beginner': 0, 'Novice': 1, 'Intermediate': 2, 'Advanced': 3}
level_reverse = {value: key for key, value in level_map.items()}

In [0]:
def combine_goals(group):
    unique_goals = sorted({g for sublist in group for g in sublist})
    return str([str(g) for g in unique_goals])

def combine_levels(group):
    mapped = group.explode().map(level_map)
    lo, hi = int(mapped.min()), int(mapped.max())
    return str([level_reverse[n] for n in range(lo, hi + 1)])

prog['goal'] = prog.groupby('title')['goal'].transform(combine_goals)
prog['level'] = prog.groupby('title')['level'].transform(combine_levels)

# Create a new summary sheet

In [0]:
group_keys = ["title", "week", "day"]

# count how often each exercise_label shows up within each day
counts = prog.groupby(group_keys + ["exercise_label"]).size().reset_index(name="n")

max_n = counts.groupby(group_keys)["n"].transform("max")
top_labels = counts[counts["n"] == max_n]

day_label = (
    top_labels.sort_values("exercise_label")
    .groupby(group_keys)["exercise_label"]
    .apply(lambda s: ", ".join(s))   # e.g. "Back, Legs" on ties
    .reset_index(name="day_label")
)

prog = prog.merge(day_label, on=group_keys, how="left")

In [0]:
day_label

# Program Summary Sheet (include the workout days)

In [0]:
titles = prog.title.unique()

records = []
groups = dict(tuple(prog.groupby('title')))

for title in titles:
    tmp = groups[title]

    desc_notna = tmp.description.dropna()
    description = desc_notna.iloc[0] if not desc_notna.empty else 'N/A'

    tmp.level = tmp.level.apply(ast.literal_eval)
    tmp.goal = tmp.goal.apply(ast.literal_eval)
    tmp.day_label = tmp.day_label.apply(ast.literal_eval)

    level_set = list(set().union(*tmp.level.dropna().values))
    goal_set = list(set().union(*tmp.goal.dropna().values))
    day_set = list(set().union(*tmp.day_label.dropna().values))

    last_week = int(tmp.week.max())

    day_max_per_week = tmp.groupby('week')['day'].max()
    intensity_per_day = tmp.groupby(['week', 'day'])['intensity'].mean()

    row = {
        'title': title,
        'description': description,
        'level': str(level_set),
        'goal': str(goal_set),
        'equipment': tmp.equipment.values[0],
        'program_length_weeks': last_week,
        'time_per_workout': tmp.time_per_workout.values[0],
        'total_exercises': len(tmp),
        'program_length_days': int(day_max_per_week.loc[1:last_week].sum()),
        'average_intensity': round(intensity_per_day.mean(), 2),
        'workout_days': str(day_set),
    }
    records.append(row)

df3 = pd.DataFrame(records)

In [0]:
df3.to_csv(DATA_DIR / 'program_summary_v3_2.csv') # Version 3.2 post november cleaning updates